# Relative Cost Estimation — Evaluation

Evaluate the **Embedding + KNN** approach for estimating relative cost ratios across models.

- **Method**: For a given query, embed it, find K nearest neighbors in the training set, and use their cost ratios (relative to a reference model) as the estimate.
- **Evaluation**: Standard train-test split, measuring MAPE and MAE.

In [26]:
import sys, os, importlib
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'ThinkingTax'))
# Also handle running from the repo root
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import method.relative_cost_estimator as _rce_mod
importlib.reload(_rce_mod)
from method.relative_cost_estimator import RelativeCostEstimator

## 1. Configuration

In [ ]:
DATA_DIR = "../data/consolidated"
REFERENCE_MODEL = "MiniMax-M2.5"
EMBEDDING_PROVIDER = "gemini"
K = 5
TEST_RATIO = 0.2
SEED = 42

# Load API key
with open("../apikey.json") as f:
    api_keys = json.load(f)

API_KEY = api_keys["gemini_api_key"]

# Load models from experiment config (only use these 8 models)
with open("../constant/experiment_config.json") as f:
    exp_config = json.load(f)

MODELS = [m["model_name"] for m in exp_config["models"]]
DATASETS = [d["file_prefix"] for d in exp_config["datasets"]]

# Load model pricing info
with open("../constant/model_info.json") as f:
    model_info = json.load(f)

MODEL_INFO = model_info["models"]
short_names = {m["model_name"]: m["short_name"] for m in MODEL_INFO}

print(f"Using {len(MODELS)} models: {MODELS}")
print(f"Using {len(DATASETS)} datasets: {DATASETS}")

Using 8 models: ['gpt-5.2-high', 'gpt-5-mini', 'gemini-3.1-pro-preview', 'gemini-3-flash-preview', 'claude-opus-4.6-thinking', 'claude-haiku-4.5', 'kimi-k2.5', 'MiniMax-M2.5']


## 2. Build Index

In [ ]:
# Debug: check data coverage
from pathlib import Path
from collections import Counter

_data_dir = Path(DATA_DIR)
_candidates = ['claude-opus-4.6-thinking','gemini-3.1-pro-preview','gemini-3-flash-preview',
    'claude-haiku-4.5','claude-opus-4.6','gpt-5.2-high','MiniMax-M2.5','gpt-5-mini','kimi-k2.5','gpt-5.2']

_query_data = {}
for _f in sorted(_data_dir.glob('*.json')):
    _stem = _f.stem
    _model_name = _ds_prefix = None
    for _m in _candidates:
        if _stem.endswith('-' + _m):
            _model_name, _ds_prefix = _m, _stem[:-len('-'+_m)]
            break
    if _model_name is None or _model_name not in MODELS:
        continue
    if _ds_prefix not in DATASETS:
        continue
    with open(_f) as _fh:
        _d = json.load(_fh)
    for _rec in _d.get('records', []):
        _key = (_ds_prefix, _rec['index'])
        _query_data.setdefault(_key, {'models': set(), 'costs': {}})
        _query_data[_key]['models'].add(_model_name)
        _query_data[_key]['costs'][_model_name] = _rec.get('cost', 0)

_total = len(_query_data)
_complete = sum(1 for v in _query_data.values() if len(v['models']) == len(MODELS))
_ref_ok = sum(1 for v in _query_data.values() if v['costs'].get(REFERENCE_MODEL, 0) > 0)
_both = sum(1 for v in _query_data.values() if len(v['models']) == len(MODELS) and v['costs'].get(REFERENCE_MODEL, 0) > 0)

print(f"Total unique queries: {_total}")
print(f"With all {len(MODELS)} models: {_complete}")
print(f"With {REFERENCE_MODEL} cost > 0: {_ref_ok}")
print(f"Both (used for indexing): {_both}")

_ds_counts, _ds_complete = Counter(), Counter()
for (_ds, _idx), _v in _query_data.items():
    _ds_counts[_ds] += 1
    if len(_v['models']) == len(MODELS):
        _ds_complete[_ds] += 1
print("\nPer dataset:")
for _ds in sorted(_ds_counts):
    print(f"  {_ds}: {_ds_complete[_ds]}/{_ds_counts[_ds]} complete")

_incomplete = [(k,v) for k,v in _query_data.items() if len(v['models']) < len(MODELS)]
if _incomplete:
    print(f"\nSample incomplete ({len(_incomplete)} total):")
    for (_ds, _idx), _v in _incomplete[:3]:
        print(f"  ({_ds},{_idx}): has {len(_v['models'])}, missing {set(MODELS)-_v['models']}")

Total unique queries: 12352
With all 8 models: 11975
With MiniMax-M2.5 cost > 0: 12346
Both (used for indexing): 11975

Per dataset:
  aime-hybrid: 60/60 complete
  arc-agi-v1: 400/400 complete
  arenahard-test: 748/751 complete
  gpqa-test: 198/198 complete
  hle-test: 1903/2159 complete
  livecodebench-test: 961/1056 complete
  livemathbench-test: 119/122 complete
  mmlupro-test_3000: 3000/3000 complete
  simpleqa-test: 4309/4327 complete
  tau2-test: 277/279 complete

Sample incomplete (377 total):
  (arenahard-test,683): has 7, missing {'claude-opus-4.6-thinking'}
  (arenahard-test,750): has 7, missing {'claude-opus-4.6-thinking'}
  (arenahard-test,0): has 1, missing {'gemini-3.1-pro-preview', 'gpt-5-mini', 'claude-haiku-4.5', 'gemini-3-flash-preview', 'kimi-k2.5', 'MiniMax-M2.5', 'gpt-5.2-high'}


In [ ]:
estimator = RelativeCostEstimator(
    reference_model=REFERENCE_MODEL,
    models=MODELS,
    embedding_provider=EMBEDDING_PROVIDER,
    api_key=API_KEY,
    k=K,
)
estimator.build_index(data_dir=DATA_DIR, datasets=DATASETS)
print(estimator)
print(f"Models: {estimator.model_list}")
print(f"Total indexed queries: {estimator.n_queries}")

RelativeCostEstimator(ref='MiniMax-M2.5', models=8, provider='gemini', k=5, index=built, n_queries=11753)
Models: ['gpt-5.2-high', 'gpt-5-mini', 'gemini-3.1-pro-preview', 'gemini-3-flash-preview', 'claude-opus-4.6-thinking', 'claude-haiku-4.5', 'kimi-k2.5', 'MiniMax-M2.5']
Total indexed queries: 11753


## 3. Train-Test Evaluation

In [29]:
# KNN method
mae_knn = estimator.evaluate(test_ratio=TEST_RATIO, seed=SEED, metric="mae")

# Pricing baseline
mae_baseline = estimator.pricing_baseline(MODEL_INFO, test_ratio=TEST_RATIO, seed=SEED)

eval_df = pd.DataFrame({
    "Model": [short_names.get(m, m) for m in estimator.model_list],
    "Pricing Baseline MAE": [f"{mae_baseline[m]:.4f}" for m in estimator.model_list],
    "KNN MAE": [f"{mae_knn[m]:.4f}" for m in estimator.model_list],
})
print("=== Per-Model MAE Comparison (Train-Test Split) ===")
print(f"Test ratio: {TEST_RATIO}, K: {K}, Seed: {SEED}\n")
print(eval_df.to_string(index=False))
print()
avg_baseline = np.mean(list(mae_baseline.values()))
avg_knn = np.mean(list(mae_knn.values()))
print(f"Average MAE — Pricing Baseline: {avg_baseline:.4f},  KNN: {avg_knn:.4f}")
print(f"Improvement: {(avg_baseline - avg_knn) / avg_baseline:.1%}")

=== Per-Model MAE Comparison (Train-Test Split) ===
Test ratio: 0.2, K: 5, Seed: 42

           Model Pricing Baseline MAE KNN MAE
         GPT-5.2              19.7340 23.4156
      GPT-5 Mini               1.6397  1.5873
  Gemini 3.1 Pro              55.5968 82.4009
  Gemini 3 Flash              28.2074 23.3242
 Claude Opus 4.6              27.6163 27.1259
Claude Haiku 4.5               2.9995  1.0823
       Kimi K2.5               9.1206  6.6080
    MiniMax-M2.5               0.0000  0.0000

Average MAE — Pricing Baseline: 18.1143,  KNN: 20.6930
Improvement: -14.2%


## 4. Sensitivity Analysis: Varying K

In [21]:
k_values = [1, 3, 5, 10, 20, 50]
k_mape = {}

for k_val in k_values:
    estimator.k = k_val
    result = estimator.evaluate(test_ratio=TEST_RATIO, seed=SEED, metric="mape")
    k_mape[k_val] = np.mean(list(result.values()))

# Restore original K
estimator.k = K

k_df = pd.DataFrame({"K": list(k_mape.keys()), "Avg MAPE": [f"{v:.2%}" for v in k_mape.values()]})
print("=== K Sensitivity ===")
print(k_df.to_string(index=False))

=== K Sensitivity ===
 K Avg MAPE
 1  259.00%
 3  251.22%
 5  257.92%
10  279.39%
20  299.61%
50  338.74%


## 5. Sensitivity Analysis: Varying Test Ratio

In [22]:
test_ratios = [0.1, 0.2, 0.3, 0.5]
ratio_mape = {}

for tr in test_ratios:
    result = estimator.evaluate(test_ratio=tr, seed=SEED, metric="mape")
    ratio_mape[tr] = np.mean(list(result.values()))

ratio_df = pd.DataFrame({
    "Test Ratio": list(ratio_mape.keys()),
    "Avg MAPE": [f"{v:.2%}" for v in ratio_mape.values()],
})
print("=== Test Ratio Sensitivity ===")
print(ratio_df.to_string(index=False))

=== Test Ratio Sensitivity ===
 Test Ratio Avg MAPE
        0.1  258.89%
        0.2  257.92%
        0.3  264.94%
        0.5  277.57%


## 6. Per-Query Error Distribution

In [30]:
# Per-query MAE for both methods
train_idx, test_idx = estimator.train_test_split(TEST_RATIO, SEED)

train_embs = estimator._embeddings[train_idx]
train_ratios = estimator._cost_ratios[train_idx]
test_embs = estimator._embeddings[test_idx]
test_ratios = estimator._cost_ratios[test_idx]

# --- KNN per-query MAE ---
sim_matrix = estimator._cosine_similarity(test_embs, train_embs)
k = min(estimator.k, len(train_idx))

per_query_mae_knn = []
for i in range(len(test_idx)):
    sims = sim_matrix[i]
    top_k_idx = np.argsort(sims)[-k:][::-1]
    top_k_sims = sims[top_k_idx]
    weights = top_k_sims / top_k_sims.sum()
    predicted = np.average(train_ratios[top_k_idx], axis=0, weights=weights)
    actual = test_ratios[i]
    mae = np.mean(np.abs(predicted - actual))
    per_query_mae_knn.append(mae)

per_query_mae_knn = np.array(per_query_mae_knn)

# --- Pricing baseline per-query MAE ---
pricing = {m["model_name"]: m["input_price_per_MTok"] + m["output_price_per_MTok"] for m in MODEL_INFO}
ref_price = pricing[REFERENCE_MODEL]
baseline_ratios = np.array([pricing[m] / ref_price for m in estimator.model_list])
per_query_mae_baseline = np.mean(np.abs(test_ratios - baseline_ratios[None, :]), axis=1)

print(f"Per-query MAE statistics (n={len(test_idx)}):")
print(f"{'':>10s}  {'Baseline':>10s}  {'KNN':>10s}")
print(f"{'Mean':>10s}  {per_query_mae_baseline.mean():10.4f}  {per_query_mae_knn.mean():10.4f}")
print(f"{'Median':>10s}  {np.median(per_query_mae_baseline):10.4f}  {np.median(per_query_mae_knn):10.4f}")
print(f"{'Std':>10s}  {per_query_mae_baseline.std():10.4f}  {per_query_mae_knn.std():10.4f}")
print(f"{'P90':>10s}  {np.percentile(per_query_mae_baseline, 90):10.4f}  {np.percentile(per_query_mae_knn, 90):10.4f}")
print(f"{'P95':>10s}  {np.percentile(per_query_mae_baseline, 95):10.4f}  {np.percentile(per_query_mae_knn, 95):10.4f}")

Per-query MAE statistics (n=2350):
              Baseline         KNN
      Mean     18.1143     20.6930
    Median      8.3260     12.0760
       Std     33.6008     33.2978
       P90     39.0668     39.2789
       P95     49.5643     69.0167


## 7. Per-Dataset Breakdown

In [31]:
# Per-dataset MAE breakdown for both methods
test_keys = [estimator._query_keys[i] for i in test_idx]
datasets_in_test = sorted(set(k[0] for k in test_keys))

rows = []
for ds in datasets_in_test:
    mask = np.array([k[0] == ds for k in test_keys])
    n = int(mask.sum())
    bl_mae = per_query_mae_baseline[mask].mean()
    knn_mae = per_query_mae_knn[mask].mean()
    rows.append({"Dataset": ds, "N": n,
                 "Baseline MAE": f"{bl_mae:.4f}",
                 "KNN MAE": f"{knn_mae:.4f}"})

ds_df = pd.DataFrame(rows)
print("=== Per-Dataset MAE Comparison ===")
print(ds_df.to_string(index=False))

=== Per-Dataset MAE Comparison ===
           Dataset   N Baseline MAE KNN MAE
       aime-hybrid  16      10.5166 18.6639
        arc-agi-v1  82      23.0218 15.2430
    arenahard-test 147      12.5092 17.3481
         gpqa-test  36      11.4328 11.5433
          hle-test 379      29.7641 22.7008
livecodebench-test 199      20.5526 16.2652
livemathbench-test  17       7.6493 12.6976
 mmlupro-test_3000 587      10.3699 12.5507
     simpleqa-test 833      18.6081 28.6523
         tau2-test  54      21.7387 15.2496


In [32]:
# LaTeX Table: Per-Model MAE comparison (Pricing Baseline vs KNN)

lines = []
lines.append(r"\begin{table}[t]")
lines.append(r"\centering")
lines.append(r"\caption{MAE of relative cost estimation per model (reference: "
             + short_names.get(REFERENCE_MODEL, REFERENCE_MODEL)
             + r", $K{=}" + str(K) + r"$, test ratio$=$" + str(TEST_RATIO) + r").}")
lines.append(r"\label{tab:relative-cost-mae}")
lines.append(r"\begin{tabular}{lcc}")
lines.append(r"\toprule")
lines.append(r"Model & Pricing Baseline & Embedding + KNN \\")
lines.append(r"\midrule")
for model in estimator.model_list:
    name = short_names.get(model, model)
    bl = mae_baseline[model]
    knn = mae_knn[model]
    # Bold the better (lower) value
    if model == REFERENCE_MODEL:
        lines.append(f"\\textbf{{{name}}} & 0.00 & 0.00 \\\\")
    elif bl <= knn:
        lines.append(f"{name} & \\textbf{{{bl:.2f}}} & {knn:.2f} \\\\")
    else:
        lines.append(f"{name} & {bl:.2f} & \\textbf{{{knn:.2f}}} \\\\")
lines.append(r"\midrule")
avg_bl = np.mean(list(mae_baseline.values()))
avg_knn = np.mean(list(mae_knn.values()))
if avg_bl <= avg_knn:
    lines.append(f"Average & \\textbf{{{avg_bl:.2f}}} & {avg_knn:.2f} \\\\")
else:
    lines.append(f"Average & {avg_bl:.2f} & \\textbf{{{avg_knn:.2f}}} \\\\")
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table1 = "\n".join(lines)
print(latex_table1)

print("\n" + "="*60 + "\n")

# LaTeX Table 2: Per-Dataset MAE comparison
lines2 = []
lines2.append(r"\begin{table}[t]")
lines2.append(r"\centering")
lines2.append(r"\caption{Per-dataset MAE of relative cost estimation.}")
lines2.append(r"\label{tab:relative-cost-mae-dataset}")
lines2.append(r"\begin{tabular}{lccc}")
lines2.append(r"\toprule")
lines2.append(r"Dataset & $N$ & Pricing Baseline & Embedding + KNN \\")
lines2.append(r"\midrule")
total_n = 0
for ds in datasets_in_test:
    mask = np.array([k[0] == ds for k in test_keys])
    n = int(mask.sum())
    total_n += n
    bl_val = per_query_mae_baseline[mask].mean()
    knn_val = per_query_mae_knn[mask].mean()
    if bl_val <= knn_val:
        lines2.append(f"{ds} & {n} & \\textbf{{{bl_val:.2f}}} & {knn_val:.2f} \\\\")
    else:
        lines2.append(f"{ds} & {n} & {bl_val:.2f} & \\textbf{{{knn_val:.2f}}} \\\\")
lines2.append(r"\midrule")
overall_bl = per_query_mae_baseline.mean()
overall_knn = per_query_mae_knn.mean()
if overall_bl <= overall_knn:
    lines2.append(f"Overall & {total_n} & \\textbf{{{overall_bl:.2f}}} & {overall_knn:.2f} \\\\")
else:
    lines2.append(f"Overall & {total_n} & {overall_bl:.2f} & \\textbf{{{overall_knn:.2f}}} \\\\")
lines2.append(r"\bottomrule")
lines2.append(r"\end{tabular}")
lines2.append(r"\end{table}")

latex_table2 = "\n".join(lines2)
print(latex_table2)

# Save to file
with open("../figure/relative_cost_tables.tex", "w") as f:
    f.write("% Table 1: Per-Model MAE Comparison\n")
    f.write(latex_table1)
    f.write("\n\n")
    f.write("% Table 2: Per-Dataset MAE Comparison\n")
    f.write(latex_table2)
print("\nSaved to figure/relative_cost_tables.tex")

\begin{table}[t]
\centering
\caption{MAE of relative cost estimation per model (reference: MiniMax-M2.5, $K{=}5$, test ratio$=$0.2).}
\label{tab:relative-cost-mae}
\begin{tabular}{lcc}
\toprule
Model & Pricing Baseline & Embedding + KNN \\
\midrule
GPT-5.2 & \textbf{19.73} & 23.42 \\
GPT-5 Mini & 1.64 & \textbf{1.59} \\
Gemini 3.1 Pro & \textbf{55.60} & 82.40 \\
Gemini 3 Flash & 28.21 & \textbf{23.32} \\
Claude Opus 4.6 & 27.62 & \textbf{27.13} \\
Claude Haiku 4.5 & 3.00 & \textbf{1.08} \\
Kimi K2.5 & 9.12 & \textbf{6.61} \\
\textbf{MiniMax-M2.5} & 0.00 & 0.00 \\
\midrule
Average & \textbf{18.11} & 20.69 \\
\bottomrule
\end{tabular}
\end{table}


\begin{table}[t]
\centering
\caption{Per-dataset MAE of relative cost estimation.}
\label{tab:relative-cost-mae-dataset}
\begin{tabular}{lccc}
\toprule
Dataset & $N$ & Pricing Baseline & Embedding + KNN \\
\midrule
aime-hybrid & 16 & \textbf{10.52} & 18.66 \\
arc-agi-v1 & 82 & 23.02 & \textbf{15.24} \\
arenahard-test & 147 & \textbf{12.51} & 1